# Week 21, Day 4: Git: branches and merging

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w021d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
parents = {"c1": None, "c2": "c1", "c3": "c2", "x1": "c2"}
branches = {"main": "c3", "idea": "x1"}

def history(commit):
    out = []
    while commit:
        out.append(commit)
        commit = parents[commit]
    return out

print(history(branches["main"]))
print(history(branches["idea"]))

*Interactive figure on the website: Branches in the practice terminal. Watch the files change as you switch, and use git log --oneline to see where each branch points..*

> **Key idea.** A branch is a name pointing at a commit; HEAD is where you are. `git switch -c name` starts a branch, `git switch name` moves between them, and `git merge name` brings another branch's work in: a fast-forward if your branch hasn't moved, otherwise a three-way merge. A conflict means both changed the same lines: edit the file, remove the markers, `git add`, `git commit`.

### Exercise: Committing moves the branch

A repository is modelled as `parents` (commit ID to its parent, or `None`), `branches` (branch name to commit ID) and `head` (the current branch's name). Write `commit(repo, cid)` that records a new commit with ID `cid` whose parent is the current branch's commit, and moves the current branch to it. Then `create_branch(repo, name)` (at the current commit) and `switch(repo, name)`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    repo = {\"parents\": {}, \"branches\": {\"main\": None}, \"head\": \"main\"}\n    commit(repo, \"c1\")\n    commit(repo, \"c2\")\n    create_branch(repo, \"idea\")\n    switch(repo, \"idea\")\n    commit(repo, \"x1\")\n    switch(repo, \"main\")\n    commit(repo, \"c3\")\n    assert repo[\"branches\"] == {\"main\": \"c3\", \"idea\": \"x1\"}\n    assert repo[\"parents\"] == {\"c1\": None, \"c2\": \"c1\", \"x1\": \"c2\", \"c3\": \"c2\"}\n    assert repo[\"head\"] == \"main\"")

### Exercise: Fast-forward or not?

With `parents` as in the last exercise (but a merge commit may have **two** parents, so here each value is a **list** of parent IDs), write `ancestors(parents, cid)` returning the set of every commit reachable from `cid` (including itself), and `merge_kind(parents, ours, theirs)` returning `"up to date"` if `theirs` is already in our history, `"fast-forward"` if `ours` is in `theirs`'s history, and `"three-way"` otherwise.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    p = {\"c1\": [], \"c2\": [\"c1\"], \"c3\": [\"c2\"], \"x1\": [\"c2\"], \"x2\": [\"x1\"], \"m\": [\"c3\", \"x2\"]}\n    assert ancestors(p, \"c3\") == {\"c1\", \"c2\", \"c3\"} and ancestors(p, \"m\") == {\"c1\", \"c2\", \"c3\", \"x1\", \"x2\", \"m\"}\n    assert merge_kind(p, \"c2\", \"x2\") == \"fast-forward\"\n    assert merge_kind(p, \"c3\", \"x2\") == \"three-way\"\n    assert merge_kind(p, \"m\", \"x1\") == \"up to date\" and merge_kind(p, \"c3\", \"c3\") == \"up to date\"")

### Exercise: The merge base

Write `merge_base(parents, a, b)` returning the newest commit that both `a` and `b` have in their history: search outwards from `b` in breadth-first order (nearest first, with a `deque`), and return the first commit that is among `a`'s ancestors. Use your `ancestors` from the last exercise (it's in the starter).

In [ ]:
from collections import deque

def ancestors(parents, cid):
    seen = set()
    stack = [cid]
    while stack:
        c = stack.pop()
        if c not in seen:
            seen.add(c)
            stack.extend(parents[c])
    return seen

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    p = {\"c1\": [], \"c2\": [\"c1\"], \"c3\": [\"c2\"], \"x1\": [\"c2\"], \"x2\": [\"x1\"], \"c4\": [\"c3\"]}\n    assert merge_base(p, \"c4\", \"x2\") == \"c2\" and merge_base(p, \"x2\", \"c4\") == \"c2\"\n    assert merge_base(p, \"c3\", \"c1\") == \"c1\" and merge_base(p, \"x1\", \"x1\") == \"x1\"\n    q = {\"a\": [], \"b\": [\"a\"], \"c\": [\"a\"], \"m\": [\"b\", \"c\"], \"d\": [\"c\"]}\n    assert merge_base(q, \"m\", \"d\") == \"c\"")

### Exercise: A three-way merge of files

Write `merge_files(base, ours, theirs)`, where each is a dictionary from file name to its contents (a missing name means the file doesn't exist). Return `(merged, conflicts)`: for each file, if both sides have the same contents, keep it; if only one side changed it from `base`, take that side (and if that side deleted it, leave it out); if both changed it differently, it's a conflict: put `"<<<<<<< ours\n" + ours_text + "=======\n" + theirs_text + ">>>>>>> theirs\n"` in `merged` (using `""` for a missing side) and add its name to `conflicts`, a sorted list.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    base = {\"a.txt\": \"1\\n\", \"b.txt\": \"1\\n\", \"c.txt\": \"1\\n\", \"d.txt\": \"1\\n\"}\n    ours = {\"a.txt\": \"2\\n\", \"b.txt\": \"1\\n\", \"c.txt\": \"ours\\n\", \"e.txt\": \"new\\n\"}\n    theirs = {\"a.txt\": \"1\\n\", \"b.txt\": \"3\\n\", \"c.txt\": \"theirs\\n\", \"d.txt\": \"1\\n\"}\n    merged, conflicts = merge_files(base, ours, theirs)\n    assert conflicts == [\"c.txt\"]\n    assert merged[\"a.txt\"] == \"2\\n\" and merged[\"b.txt\"] == \"3\\n\" and merged[\"e.txt\"] == \"new\\n\"\n    assert \"d.txt\" not in merged\n    assert merged[\"c.txt\"] == \"<<<<<<< ours\\nours\\n=======\\ntheirs\\n>>>>>>> theirs\\n\"\n    assert merge_files({}, {\"x\": \"1\"}, {\"x\": \"1\"}) == ({\"x\": \"1\"}, [])")

### Exercise: Read the conflict markers

Write `sides(text)` that takes a file with **one** conflict in it and returns `(ours_version, theirs_version)`: the whole file as it would be with our side chosen, and with theirs, as strings with the lines that aren't in the conflict kept on both. Then `has_markers(text)` returning `True` if any line starts with `<<<<<<<`, `=======` or `>>>>>>>`, which is how you check you've cleaned up before committing.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    t = \"Welcome\\n<<<<<<< main\\nOpen 9 to 5\\n=======\\nOpen Mon to Fri\\n>>>>>>> hours\\nFind us in Leeds\\n\"\n    assert sides(t) == (\"Welcome\\nOpen 9 to 5\\nFind us in Leeds\\n\", \"Welcome\\nOpen Mon to Fri\\nFind us in Leeds\\n\")\n    assert has_markers(t) and not has_markers(\"Welcome\\nOpen 9 to 5\\n\")\n    assert not has_markers(\"a == b\\n\")")

### Bug hunt: The branch didn't move

After committing on `idea`, this model still has `idea` pointing at the old commit, because `commit` moves `main` whatever branch is current. Fix it so it prints `{'main': 'c1', 'idea': 'x1'}`.

In [ ]:
repo = {"parents": {"c1": None}, "branches": {"main": "c1", "idea": "c1"}, "head": "idea"}

def commit(repo, cid):
    repo["parents"][cid] = repo["branches"]["main"]
    repo["branches"]["main"] = cid

commit(repo, "x1")
print(repo["branches"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"{'main': 'c1', 'idea': 'x1'}\")")

### Bug hunt: A resolved file that isn't

Someone "resolved" a conflict by deleting only the `=======` line. This checker should catch it and print `still has markers`, but it only looks for `=======`. Fix `has_markers` so it catches any of the three marker lines.

In [ ]:
text = "Welcome\n<<<<<<< main\nOpen 9 to 5\nOpen Mon to Fri\n>>>>>>> hours\n"

def has_markers(text):
    return any(line.startswith("=======") for line in text.splitlines())

print("still has markers" if has_markers(text) else "clean")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"caught\"\n    expect_output(\"still has markers\")")

### Puzzle: A feature branch

Arrange the commands for building a feature on its own branch and merging it into main. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # True or False: creating a branch copies all the files
b = ___   # the kind of merge when your branch hasn't moved since the other split off (two words with a hyphen)
c = ___   # how many parents a merge commit has
d = ___   # after editing a conflicted file, the command that marks it as resolved (two words)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a is False and b == \"fast-forward\" and c == 2 and d == \"git add\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: draw the history

`git log --oneline --graph` draws history as text. Write `graph(parents, messages, tip)` that lists every commit reachable from `tip`, newest first, as lines `"<id> <message>"`, where "newest first" means: a commit always comes **before** all of its parents (a topological order). When several commits are ready, take them in the order you find them: start from `tip`, and use a depth-first approach that only outputs a commit once all commits that have it as a parent (within the history from `tip`) have been output. Merge commits get `" (merge)"` after their message.

Then write `branch_points(parents, tip)`: a sorted list of commits (reachable from `tip`) that are a parent of **two or more** commits, which is where history split.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("P = {\"c1\": [], \"c2\": [\"c1\"], \"c3\": [\"c2\"], \"x1\": [\"c2\"], \"x2\": [\"x1\"], \"m\": [\"c3\", \"x2\"], \"c4\": [\"m\"]}\nM = {\"c1\": \"Start\", \"c2\": \"Add page\", \"c3\": \"Fix typo\", \"x1\": \"Try idea\", \"x2\": \"Finish idea\", \"m\": \"Merge idea\", \"c4\": \"Tidy\"}\ndef test_graph():\n    \"graph\"\n    lines = graph(P, M, \"c4\")\n    assert lines[0] == \"c4 Tidy\" and lines[1] == \"m Merge idea (merge)\" and lines[-1] == \"c1 Start\"\n    assert len(lines) == 7\n    order = [l.split()[0] for l in lines]\n    for c, ps in P.items():\n        for p in ps:\n            assert order.index(c) < order.index(p), f\"{c} must come before its parent {p}\"\n    assert graph(P, M, \"x1\") == [\"x1 Try idea\", \"c2 Add page\", \"c1 Start\"]\ndef test_points():\n    \"branch_points\"\n    assert branch_points(P, \"c4\") == [\"c2\"] and branch_points(P, \"x2\") == []")

## Warm-up

Two branches, sharing their first two commits and then going separate ways. Each branch is just a **name** pointing at a commit; its history is whatever you reach by following parents back from there. That really is all a branch is.

## Branches are names

A **branch** is a name for a commit, and it moves forward automatically every time you commit on it. When you `git init`, you get one branch, usually called `main`.

**HEAD** is Git's name for "where you are now": normally it points at the branch you're on, and that branch points at its newest commit. When you commit, the new commit's parent is the current one, and the branch name moves to the new commit.

Creating a branch makes a new name pointing at the **current** commit. Nothing is copied: creating a branch is instant, whatever the size of the project.

```
          c3  <- main
         /
c1 -- c2
         \
          x1  <- idea  <- HEAD
```

| Command | Does |
|---|---|
| `git branch` | list branches (`*` marks the current one) |
| `git branch name` | create a branch at the current commit |
| `git switch name` | move to a branch: your files change to match its newest commit |
| `git switch -c name` | create a branch and switch to it in one go |
| `git branch -d name` | delete a branch that's been merged |
| `git log --oneline --graph --all` | draw every branch's history |

(Older tutorials use `git checkout name` and `git checkout -b name`; they still work, but `switch` was added to make things clearer.)

## Switching

When you switch branches, Git changes the files in your folder to match the commit that branch points at. Files that exist on one branch and not the other appear and disappear. That's why Git refuses to switch if you have uncommitted changes that would be overwritten: commit them first (or set them aside with `git stash`, which you'll meet later).

Try it in the practice terminal. The tasks walk through a full feature branch: make it, commit on it, switch back, see your files change, and merge.

## Merging

`git merge other` brings the work from branch `other` into the branch you're on. There are two kinds of merge:

**Fast-forward.** If your branch hasn't moved since `other` split off, then `other` is just your branch with more commits on top. Git simply moves your branch name forward to `other`'s commit. No new commit is needed.

```
before:  c1 -- c2  <- main            after:  c1 -- c2 -- x1 -- x2  <- main, other
                 \
                  x1 -- x2  <- other
```

**Three-way merge.** If both branches have new commits, Git finds their **merge base** (the newest commit they share) and compares each branch with it, file by file and line by line:

- a part changed on only one side: take that side's change;
- a part changed on both sides in the same way: take it;
- a part changed on both sides **differently**: that's a **conflict**, and you decide.

If there are no conflicts, Git makes a **merge commit** with two parents, joining the two lines of history.

## Conflicts

When both branches changed the same lines, Git stops, writes **both** versions into the file between markers, and asks you to choose:

```
Welcome to our site
<<<<<<< main
Open every day, 9 to 5
=======
Open Monday to Friday
>>>>>>> hours
Find us in Leeds
```

Above the `=======` is your branch's version (`main`); below it, the other branch's (`hours`). To resolve it:

1. Open the file and edit it to what it **should** say: one version, the other, or a mix. Delete all three marker lines.
2. `git add` the file, to mark the conflict as resolved.
3. `git commit` to finish the merge.

`git status` lists the files that still need resolving. If you'd rather not deal with it now, `git merge --abort` puts everything back as it was before the merge.

Conflicts aren't errors; they're Git being careful. They're less likely when branches are short-lived and each one changes one thing.

## A branching workflow

A simple, widely used way of working, alone or in a team:

1. Keep `main` working at all times.
2. For each change (a feature, a fix), `git switch -c` a branch with a descriptive name: `add-search`, `fix-empty-file`.
3. Commit on the branch as you go.
4. When it's done and tested, switch to `main` and merge it (in a team, through a pull request: tomorrow).
5. Delete the branch: `git branch -d add-search`.

## Practice

These exercises model branches and merging in Python, the way Git does it inside.

## Quiz

## Daily challenge

## Before you go

On your own computer, in your `atlas` repository: make a branch, change a file and commit, switch back to `main`, change the **same** line differently and commit, then merge, so you meet a real conflict on purpose. Resolve it and look at `git log --oneline --graph --all`. Tomorrow: GitHub, where your repositories live online, and pull requests, how teams review each other's work.